In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("DemoSparkApp").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 05:21:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data = [1,2,3,4,5]

In [4]:
rdd = spark.sparkContext.parallelize(data)

26/06/12 05:21:29 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
for element in rdd.collect():
    print(element)

1
2
3
4
5


In [6]:
spark.stop()

In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (1, "Alice", "Engineering", 75000, 25),
    (2, "Bob", "Marketing", 60000, 30),
    (3, "Charlie", "Engineering", 80000, 35),
    (4, "David", "Sales", 65000, 28),
    (5, "Eve", "Marketing", 70000, 32)
]
columns = ["Id", "Name", "Department", "Salary", "Age"]
df = spark.createDataFrame(data, columns);

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 03:48:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_select = df.select("Name","Department","Salary")

In [3]:
df_select.show()

+-------+-----------+------+
|   Name| Department|Salary|
+-------+-----------+------+
|  Alice|Engineering| 75000|
|    Bob|  Marketing| 60000|
|Charlie|Engineering| 80000|
|  David|      Sales| 65000|
|    Eve|  Marketing| 70000|
+-------+-----------+------+



In [4]:
df_filter = df.filter(col("salary")>65000)

In [5]:
df_filter.show()

+---+-------+-----------+------+---+
| Id|   Name| Department|Salary|Age|
+---+-------+-----------+------+---+
|  1|  Alice|Engineering| 75000| 25|
|  3|Charlie|Engineering| 80000| 35|
|  5|    Eve|  Marketing| 70000| 32|
+---+-------+-----------+------+---+



In [6]:
df_with_col = df.withColumn("Bonus",col("Salary")*0.10)
df_with_col.show()

+---+-------+-----------+------+---+------+
| Id|   Name| Department|Salary|Age| Bonus|
+---+-------+-----------+------+---+------+
|  1|  Alice|Engineering| 75000| 25|7500.0|
|  2|    Bob|  Marketing| 60000| 30|6000.0|
|  3|Charlie|Engineering| 80000| 35|8000.0|
|  4|  David|      Sales| 65000| 28|6500.0|
|  5|    Eve|  Marketing| 70000| 32|7000.0|
+---+-------+-----------+------+---+------+



In [7]:
df.write.mode("overwrite").csv("output/employeetable",header = True)

In [8]:
df.toPandas().to_csv(
    "output/employee123.csv",
    index = False
)

26/06/13 03:48:38 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [12]:
df = spark.read \
    .option("header", True) \
    .csv("output/employee123.csv")

df.show()

+---+-------+-----------+------+---+
| Id|   Name| Department|Salary|Age|
+---+-------+-----------+------+---+
|  1|  Alice|Engineering| 75000| 25|
|  2|    Bob|  Marketing| 60000| 30|
|  3|Charlie|Engineering| 80000| 35|
|  4|  David|      Sales| 65000| 28|
|  5|    Eve|  Marketing| 70000| 32|
+---+-------+-----------+------+---+



In [10]:
# Register Temp View
df.createOrReplaceTempView("employees")
# Query with standard SQL
sql_df = spark.sql('''
 SELECT Department, AVG(Salary) as Avg_Salary
 FROM employees
 GROUP BY Department
 ORDER BY Avg_Salary DESC
''')
sql_df.show()

[Stage 11:=====================>                                    (3 + 5) / 8]

+-----------+----------+
| Department|Avg_Salary|
+-----------+----------+
|Engineering|   77500.0|
|  Marketing|   65000.0|
|      Sales|   65000.0|
+-----------+----------+



In [11]:
result = spark.sql("""
SELECT * from employees
where Salary > 70000""").show()

+---+-------+-----------+------+---+
| Id|   Name| Department|Salary|Age|
+---+-------+-----------+------+---+
|  1|  Alice|Engineering| 75000| 25|
|  3|Charlie|Engineering| 80000| 35|
+---+-------+-----------+------+---+



In [86]:
df = spark.range(0,10000000).withColumn("value",col("id")%1000)

In [88]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [27]:
print("before partitionings:",df.rdd.getNumPartitions())

before partitionings: 8


In [30]:
df_repartition = df.repartition(10)

In [32]:
print("After partition:",df_repartition.rdd.getNumPartitions())

After partition: 10


In [34]:
df_colesced = df_repartition.coalesce(2)

In [36]:
print("after coalesced:",df_colesced.rdd.getNumPartitions())

[Stage 22:>                                                         (0 + 8) / 8]

after coalesced: 2


In [37]:
optimized_df = df.filter(col("value")>500).filter(col("id")<500000)

In [38]:
optimized_df.show(10)

+---+-----+
| id|value|
+---+-----+
|501|  501|
|502|  502|
|503|  503|
|504|  504|
|505|  505|
|506|  506|
|507|  507|
|508|  508|
|509|  509|
|510|  510|
+---+-----+
only showing top 10 rows



In [91]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (101, "Aarav", "HR", 45000),
    (102, "Diya", "IT", 60000),
    (103, "Vivaan", "Finance", 55000),
    (104, "Anaya", "Marketing", 50000),
    (105, "Aditya", "IT", 65000),
    (106, "Ishita", "HR", 47000),
    (107, "Krishna", "Sales", 52000),
    (108, "Meera", "Finance", 58000),
    (109, "Arjun", "IT", 70000),
    (110, "Riya", "Marketing", 49000),
    (111, "Kabir", "Sales", 54000),
    (112, "Saanvi", "HR", 46000),
    (113, "Reyansh", "IT", 72000),
    (114, "Aadhya", "Finance", 61000),
    (115, "Atharv", "Sales", 53000),
    (116, "Myra", "Marketing", 51000),
    (117, "Shaurya", "IT", 68000),
    (118, "Pari", "HR", 48000),
    (119, "Yash", "Finance", 59000),
    (120, "Kiara", "Sales", 56000)
]
columns = ["Id", "Name", "Department", "Salary"]
df = spark.createDataFrame(data, columns);

In [92]:
df.show()

+---+-------+----------+------+
| Id|   Name|Department|Salary|
+---+-------+----------+------+
|101|  Aarav|        HR| 45000|
|102|   Diya|        IT| 60000|
|103| Vivaan|   Finance| 55000|
|104|  Anaya| Marketing| 50000|
|105| Aditya|        IT| 65000|
|106| Ishita|        HR| 47000|
|107|Krishna|     Sales| 52000|
|108|  Meera|   Finance| 58000|
|109|  Arjun|        IT| 70000|
|110|   Riya| Marketing| 49000|
|111|  Kabir|     Sales| 54000|
|112| Saanvi|        HR| 46000|
|113|Reyansh|        IT| 72000|
|114| Aadhya|   Finance| 61000|
|115| Atharv|     Sales| 53000|
|116|   Myra| Marketing| 51000|
|117|Shaurya|        IT| 68000|
|118|   Pari|        HR| 48000|
|119|   Yash|   Finance| 59000|
|120|  Kiara|     Sales| 56000|
+---+-------+----------+------+



In [75]:
df.write.mode("overwrite").csv("output/employee_data",header = True)

In [76]:
df.toPandas().to_csv(
    "output/employeetable2.csv",
    index = False
)

In [77]:
df.take(10)

[Row(Id=101, Name='Aarav', Department='HR', Salary=45000),
 Row(Id=102, Name='Diya', Department='IT', Salary=60000),
 Row(Id=103, Name='Vivaan', Department='Finance', Salary=55000),
 Row(Id=104, Name='Anaya', Department='Marketing', Salary=50000),
 Row(Id=105, Name='Aditya', Department='IT', Salary=65000),
 Row(Id=106, Name='Ishita', Department='HR', Salary=47000),
 Row(Id=107, Name='Krishna', Department='Sales', Salary=52000),
 Row(Id=108, Name='Meera', Department='Finance', Salary=58000),
 Row(Id=109, Name='Arjun', Department='IT', Salary=70000),
 Row(Id=110, Name='Riya', Department='Marketing', Salary=49000)]

In [78]:
print("before partitionings:",df.rdd.getNumPartitions())
      

before partitionings: 8


In [80]:
df_repartition = df.repartition(5)
print("After partition:",df_repartition.rdd.getNumPartitions())

After partition: 5


In [84]:
df_colesced = df_repartition.coalesce(2)
print("after coalesced:",df_colesced.rdd.getNumPartitions())

after coalesced: 2


In [82]:
df_repartition.write.mode("overwrite").csv("output/employee_data_afterP",header = True)

In [85]:
df_colesced.write.mode("overwrite").csv("output/employee_data_colesce",header = True)

In [94]:
optimized_df = df.filter(col("Salary")>55000).filter(col("Salary")<70000)
optimized_df.show()


+---+-------+----------+------+
| Id|   Name|Department|Salary|
+---+-------+----------+------+
|102|   Diya|        IT| 60000|
|105| Aditya|        IT| 65000|
|108|  Meera|   Finance| 58000|
|114| Aadhya|   Finance| 61000|
|117|Shaurya|        IT| 68000|
|119|   Yash|   Finance| 59000|
|120|  Kiara|     Sales| 56000|
+---+-------+----------+------+



In [95]:
optimized_df.explain()

== Physical Plan ==
*(1) Filter (isnotnull(Salary#309L) AND ((Salary#309L > 55000) AND (Salary#309L < 70000)))
+- *(1) Scan ExistingRDD[Id#306L,Name#307,Department#308,Salary#309L]




In [109]:
import time
start_time = time.time()
count_first_cache = optimized_df.count()
end_time = time.time()
print(f"1. Optimized execution | count: {count_first_cache} | time: {round(end_time - start_time, 2)} seconds")

1. Optimized execution | count: 7 | time: 0.13 seconds


In [111]:
optimized_df.cache()

26/06/13 06:12:40 WARN CacheManager: Asked to cache already cached data.


DataFrame[Id: bigint, Name: string, Department: string, Salary: bigint]

In [115]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf
from pyspark.sql.types import StringType, IntegerType

In [113]:
optimized_df.unpersist()
print("\n-->Unpersisted dataframe to free up memory.")


-->Unpersisted dataframe to free up memory.


In [123]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
   (1, "Aarav", 25),
    (2, "Diya", 16),
    (3, "Vivaan", 30),
    (4, "Anaya", 14),
    (5, "Aditya", 22),
    (6, "Ishita", 17),
    (7, "Krishna", 19),
    (8, "Meera", 12),
    (9, "Arjun", 35),
    (10, "Riya", 15)
]
columns = ["Id", "Name", "Age"]
df = spark.createDataFrame(data, columns);
df.show()

+---+-------+---+
| Id|   Name|Age|
+---+-------+---+
|  1|  Aarav| 25|
|  2|   Diya| 16|
|  3| Vivaan| 30|
|  4|  Anaya| 14|
|  5| Aditya| 22|
|  6| Ishita| 17|
|  7|Krishna| 19|
|  8|  Meera| 12|
|  9|  Arjun| 35|
| 10|   Riya| 15|
+---+-------+---+



In [124]:
def can_drive(age):
    if age >=18:
        return "can drive"
    else:
        return "can't drive"

In [126]:
can_drive_udf = udf(can_drive,StringType())

In [128]:
df_method1 = df.withColumn("status",can_drive_udf(col("Age")))
print("Method 1: standard udf via dataframe api")
df_method1.show()

Method 1: standard udf via dataframe api


+---+-------+---+-----------+
| Id|   Name|Age|     status|
+---+-------+---+-----------+
|  1|  Aarav| 25|  can drive|
|  2|   Diya| 16|can't drive|
|  3| Vivaan| 30|  can drive|
|  4|  Anaya| 14|can't drive|
|  5| Aditya| 22|  can drive|
|  6| Ishita| 17|can't drive|
|  7|Krishna| 19|  can drive|
|  8|  Meera| 12|can't drive|
|  9|  Arjun| 35|  can drive|
| 10|   Riya| 15|can't drive|
+---+-------+---+-----------+

